In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

In [ ]:
# Load the preprocessed dataset
df = pd.read_csv('../DATASETS/SA_preprocessed.csv')
print(df.head())

# Check for missing values
print(df.isnull().sum())

# Use 'combined' as input text and 'target' as label
texts = df['processed_text'].tolist()
labels = df['label'].tolist()

   label                                               data  \
0      0  ["Subject: [zzzzteana] RE: Alexander  \nMartin...   
1      0  ["Subject: [zzzzteana] Moscow bomber  \nMan Th...   
2      0  ["Subject: [IRR] Klez: The Virus That  Won't D...   
3      0  ["Subject: Re: [zzzzteana] Nothing like mama u...   
4      0  ["Subject: Re: [zzzzteana] Nothing like mama u...   

                                 filename  \
0  00002.9c4069e25e1ef370c078db7ee85ff9ac   
1  00003.860e3c3cee1b42ead714c5c874fe25f7   
2  00004.864220c5b6930b209cc287c361c99af1   
3  00005.bf27cdeaf0b8c4647ecd61b1d09da613   
4  00006.253ea2f9a9cc36fa0b1129b04b806608   

                                      processed_text  
0  zzzzteana re alexander martin a posted tassos ...  
1  zzzzteana moscow bomber man threatens explosio...  
2  irr klez the virus that will not die already t...  
3  re zzzzteana nothing like mama used to make in...  
4  re zzzzteana nothing like mama used to make i ...  
label             0


In [7]:
# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Split the data
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

# Filter out non-string texts and corresponding labels
# This is necessary because 'processed_text' column had 2 missing values,
# which become float('nan') in the list and cause TypeError with the tokenizer.
train_filtered_data = [(text, label) for text, label in zip(train_texts, train_labels) if isinstance(text, str)]
train_texts = [item[0] for item in train_filtered_data]
train_labels = [item[1] for item in train_filtered_data]

val_filtered_data = [(text, label) for text, label in zip(val_texts, val_labels) if isinstance(text, str)]
val_texts = [item[0] for item in val_filtered_data]
val_labels = [item[1] for item in val_filtered_data]


# Tokenize the texts
def tokenize_function(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
# Custom Dataset class
class SpamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, train_labels)
val_dataset = SpamDataset(val_encodings, val_labels)

In [9]:
# Load pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

In [10]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.002836,0.157536
2,0.001088,0.075267
3,0.000365,0.068070


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1815, training_loss=0.11608574162724815, metrics={'train_runtime': 1681.5905, 'train_samples_per_second': 8.624, 'train_steps_per_second': 1.079, 'total_flos': 3815636524830720.0, 'train_loss': 0.11608574162724815, 'epoch': 3.0})

In [11]:
# Evaluate the model
predictions = trainer.predict(val_dataset)
preds = torch.argmax(torch.tensor(predictions.predictions), axis=1)

print('Accuracy:', accuracy_score(val_labels, preds))
print('F1 Score:', f1_score(val_labels, preds))
print('Classification Report:')
print(classification_report(val_labels, preds))

Accuracy: 0.9884201819685691
F1 Score: 0.9808743169398907
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       842
           1       0.98      0.98      0.98       367

    accuracy                           0.99      1209
   macro avg       0.99      0.99      0.99      1209
weighted avg       0.99      0.99      0.99      1209



In [12]:
# Save the model and tokenizer to a directory
model_path = './spam_filter_bert'
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f'Model saved to {model_path}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./spam_filter_bert


In [13]:
import shutil
from google.colab import files

# Zip the model directory
shutil.make_archive('spam_filter_bert', 'zip', './spam_filter_bert')

# Download the zip file
files.download('spam_filter_bert.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Convert the predictions tensor to a standard Python list
preds_list = preds.tolist()

# Collect the misclassified examples
misclassified_data = []
for text, true_label, pred_label in zip(val_texts, val_labels, preds_list):
    if true_label != pred_label:
        misclassified_data.append({
            'Text': text,
            'True Label': true_label,
            'Predicted Label': pred_label
        })

# Create a DataFrame for clean formatting in Colab
misclassified_df = pd.DataFrame(misclassified_data)

print(f"Total misclassified examples: {len(misclassified_df)} out of {len(val_labels)}\n")

# Optional: Set pandas to show the full text without truncating it
pd.set_option('display.max_colwidth', None)

# Display the dataframe (display() renders nicely in Colab)
display(misclassified_df)